# Split Generation for Medieval IE Evaluation

This notebook generates the document splits used for the medieval information extraction evaluation benchmark.

The splitter operates at document level and builds three mutually exclusive partitions:

- `dev_pool`: development examples used to provide in-context learning,
- `dev_tune`: development examples reserved for prompt and configuration tuning,
- `test`: final evaluation set.

The split construction process is fully deterministic and reproducible. All selections are controlled through fixed random seeds and stable document ordering.

The pipeline performs the following steps:

1. Load document-level JSON files generated by the corpus construction pipeline.
2. Normalize event labels and infer document-level event types and subtypes.
3. Compute lightweight document complexity statistics based on entities and event counts.
4. Identify structurally rare or heterogeneous documents that must be assigned to the test split.
5. Partition pure Purchase/Sale documents into low, medium, and high complexity groups.
6. Select reproducible development subsets using seeded random sampling.
7. Construct the final split configuration while ensuring a) no overlap between splits, b) complete document coverage, and c) deterministic reproducibility.
8. Export split definitions, summary tables, and split-specific JSON folders for downstream experiments.

The resulting splits are specifically designed to isolate rare event structures in the test set, maintain controlled complexity diversity in development subsets, and support stable and reproducible evaluation workflows.

# Setup and configuration
Loads dependencies, mounts Google Drive, and defines the global splitter configuration.

In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd

BASE_DIR = Path("..")
JSON_DIR = BASE_DIR / "data/references"
OUT_DIR = BASE_DIR / "splits"
OUT_DIR.mkdir(exist_ok=True)

PLACE_WEIGHT = 0.25
EVENT_WEIGHT = 3.0
PEOPLE_WEIGHT = 1.0

print("JSON_DIR =", JSON_DIR)
print("OUT_DIR =", OUT_DIR)

# Utility functions
General helper functions for loading JSON files and normalizing data structures.

In [ ]:
def load_json(path):
    """
    Load a JSON file from disk using UTF-8 encoding.
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def json_files_by_stem(directory):
    """
    Return a mapping between file stems and JSON file paths.
    """
    return {p.stem: p for p in Path(directory).glob("*.json")}

def ensure_list(v):
    """
    Normalize a value into list form.
    """
    if v is None:
        return []
    if isinstance(v, list):
        return v
    return [v]

# Event normalization and document profiling
Normalizes event subtypes and builds the semantic event profile for each document.

In [ ]:
def normalize_event_subtype(label):
    """
    Normalize raw event subtype labels into the controlled subtype vocabulary.
    """
    if label is None:
        return None

    s = str(label).strip().lower()

    mapping = {
        "sale": "Sale",
        "retrosale": "Retrosale",
        "lease": "Lease",
        "disembargo": "Disembargo",
        "donation": "Donation",
        "compensation": "Compensation",
        "agreement": "Agreement",
        "testimony": "Testimony",
        "production": "Production",
        "reproduction": "Reproduction",
        "activity": "Activity",
    }

    return mapping.get(s, str(label))

def subtype_to_type(subtype):
    """
    Map an event subtype to its broader event type.
    """
    if subtype in {"Sale", "Retrosale", "Lease"}:
        return "Purchase"
    if subtype in {"Disembargo", "Donation", "Compensation"}:
        return "Acquisition"
    if subtype in {"Agreement", "Testimony", "Activity"}:
        return "Activity"
    if subtype in {"Production", "Reproduction"}:
        return "Production"
    return subtype

def infer_doc_event_info(doc):
    """
    Infer document-level event types and subtypes from a JSON document.
    """
    event_types = []
    event_subtypes = []

    for ev in doc.get("events", []):
        raw_type = ev.get("type")

        if raw_type == "DocumentProduction":
            event_types.append("Production")
            raw_subtypes = ensure_list(ev.get("subtype"))

            if raw_subtypes:
                for x in raw_subtypes:
                    subtype = normalize_event_subtype(x)
                    if subtype:
                        event_subtypes.append(subtype)
            else:
                event_subtypes.append("Production")

            continue

        if raw_type == "GenericActivity":
            event_types.append("Activity")
            raw_subtypes = ensure_list(ev.get("subtype"))

            if raw_subtypes:
                for x in raw_subtypes:
                    subtype = normalize_event_subtype(x)
                    if subtype:
                        event_subtypes.append(subtype)
            else:
                event_subtypes.append("Activity")

            continue

        if raw_type in {"Purchase", "Acquisition"}:
            raw_subtypes = ensure_list(ev.get("subtype"))
            subtypes = [
                normalize_event_subtype(x)
                for x in raw_subtypes
                if normalize_event_subtype(x) is not None
            ]

            if subtypes:
                for subtype in subtypes:
                    event_subtypes.append(subtype)
                    event_types.append(subtype_to_type(subtype))
            else:
                event_types.append(raw_type)

                if raw_type == "Purchase":
                    event_subtypes.append("Sale")
                elif raw_type == "Acquisition":
                    event_subtypes.append("Disembargo")

    return {
        "event_types": sorted(set(event_types)),
        "event_subtypes": sorted(set(event_subtypes)),
    }


# Split helper functions
Helper functions used by the split generation process, including pure/mixed document detection and complexity bin assignment.

In [ ]:
def is_pure_purchase_sale(row):
    """
    Check whether a document contains only Purchase/Sale events.
    """
    return (
        set(row["event_types"]) == {"Purchase"}
        and set(row["event_subtypes"]) == {"Sale"}
    )

def is_pure_acquisition(row):
    """
    Check whether a document contains only Acquisition events.
    """
    return set(row["event_types"]) == {"Acquisition"}

def is_mixed_type(row):
    """
    Check whether a document contains multiple event types.
    """
    return len(set(row["event_types"])) > 1

def assign_complexity_bins(df, score_col="complexity_score"):
    """
    Assign low, medium, and high complexity bins to documents.

    Documents are sorted by complexity score and divided into three
    approximately equal-sized groups.
    """
    df = df.sort_values(score_col, ascending=True).reset_index(drop=True).copy()

    n = len(df)
    low_end = n // 3
    mid_end = 2 * n // 3

    df["complexity_bin"] = "high"
    df.loc[:low_end - 1, "complexity_bin"] = "low"
    df.loc[low_end:mid_end - 1, "complexity_bin"] = "medium"

    return df

# Corpus profiling
Loads all document-level JSON files and constructs the main corpus dataframe with document statistics and event metadata.

In [ ]:
files = json_files_by_stem(JSON_DIR)
rows = []

for file_stem, path in sorted(files.items()):
    doc = load_json(path)
    event_info = infer_doc_event_info(doc)
    entities = doc.get("entities", [])

    n_people = sum(1 for e in entities if e.get("type") == "Person")
    n_places = sum(1 for e in entities if e.get("type") == "Place")
    n_events = len(doc.get("events", []))

    rows.append({
        "doc_id": doc.get("document_id", file_stem),
        "file_stem": file_stem,
        "n_people": n_people,
        "n_places": n_places,
        "n_events": n_events,
        "event_types": event_info["event_types"],
        "event_subtypes": event_info["event_subtypes"],
        "complexity_score": (
            PEOPLE_WEIGHT * n_people
            + PLACE_WEIGHT * n_places
            + EVENT_WEIGHT * n_events
        ),
    })

docs_df = pd.DataFrame(rows)

docs_df["is_mixed_type"] = docs_df.apply(is_mixed_type, axis=1)
docs_df["is_pure_purchase_sale"] = docs_df.apply(is_pure_purchase_sale, axis=1)
docs_df["is_pure_acquisition"] = docs_df.apply(is_pure_acquisition, axis=1)
docs_df["has_retrosale"] = docs_df["event_subtypes"].apply(lambda xs: "Retrosale" in xs)
docs_df["has_lease"] = docs_df["event_subtypes"].apply(lambda xs: "Lease" in xs)

# Test constraints and candidate inspection
Identifies documents that must be assigned to test and displays candidate documents available for dev_pool and dev_tune.

## Documents assigned to test
Uncommon or mixed types.

In [ ]:
forced_test_df = docs_df[
    docs_df["event_types"].apply(lambda xs: "Production" in xs or "Activity" in xs)
    | docs_df["has_retrosale"]
    | docs_df["has_lease"]
    | docs_df["is_mixed_type"]
].sort_values(["doc_id"])

print("\n=== DOCUMENTS THAT MUST GO TO TEST ===")
print("Total:", len(forced_test_df))

display(
    forced_test_df[[
        "doc_id",
        "event_types",
        "event_subtypes",
        "is_mixed_type",
        "has_retrosale",
        "has_lease",
        "n_people",
        "n_places",
        "n_events",
        "complexity_score",
    ]]
)

## Available documents of type Acquisition

In [ ]:
forced_test_ids = set(forced_test_df["doc_id"])

acquisition_available_df = docs_df[
    docs_df["event_types"].apply(lambda xs: "Acquisition" in xs)
    & ~docs_df["doc_id"].isin(forced_test_ids)
].copy()

acquisition_available_df["recommended_use"] = acquisition_available_df["is_pure_acquisition"].map({
    True: "dev_pool_candidate",
    False: "test_candidate",
})

acquisition_available_df = acquisition_available_df.sort_values(
    ["recommended_use", "complexity_score", "doc_id"]
)

print("\n=== ACQUISITION AVAILABLE FOR TEST / DEV_POOL ===")
print("Total:", len(acquisition_available_df))

display(
    acquisition_available_df[[
        "doc_id",
        "recommended_use",
        "event_types",
        "event_subtypes",
        "is_pure_acquisition",
        "n_people",
        "n_places",
        "n_events",
        "complexity_score",
    ]]
)

## Available Purchase/Sale documents
Documents with type Purchase and subtype Sale, categorized in bins according to its complexity in three levels.

In [ ]:
purchase_available_df = docs_df[
    docs_df["is_pure_purchase_sale"]
    & ~docs_df["doc_id"].isin(forced_test_ids)
].copy()

purchase_available_df = assign_complexity_bins(purchase_available_df)

print("\n=== PURE PURCHASE / SALE AVAILABLE ===")
print("Total:", len(purchase_available_df))

low_df = (
    purchase_available_df[
        purchase_available_df["complexity_bin"] == "low"
    ]
    .sort_values(["doc_id"])
)

print("\n=== LOW COMPLEXITY ===")
print("Total:", len(low_df))

display(
    low_df[[
        "doc_id",
        "event_types",
        "event_subtypes",
        "n_people",
        "n_places",
        "n_events",
        "complexity_score",
    ]]
)

medium_df = (
    purchase_available_df[
        purchase_available_df["complexity_bin"] == "medium"
    ]
    .sort_values(["doc_id"])
)

print("\n=== MEDIUM COMPLEXITY ===")
print("Total:", len(medium_df))

display(
    medium_df[[
        "doc_id",
        "event_types",
        "event_subtypes",
        "n_people",
        "n_places",
        "n_events",
        "complexity_score",
    ]]
)

high_df = (
    purchase_available_df[
        purchase_available_df["complexity_bin"] == "high"
    ]
    .sort_values(["doc_id"])
)

print("\n=== HIGH COMPLEXITY ===")
print("Total:", len(high_df))

display(
    high_df[[
        "doc_id",
        "event_types",
        "event_subtypes",
        "n_people",
        "n_places",
        "n_events",
        "complexity_score",
    ]]
)

# Deterministic selection helpers
Reusable deterministic selection functions based on stable document ordering and random seeds.

In [ ]:
def select_acquisition_for_dev_pool(acquisition_available_df, random_seed=1):
    """
    Select a deterministic Acquisition document for the development pool.

    Parameters
    ----------
    acquisition_available_df : pandas.DataFrame
        Candidate Acquisition documents.
    random_seed : int
        Random seed controlling deterministic sampling.

    Returns
    -------
    str or None
        Selected document identifier.
    """
    candidates = (
        acquisition_available_df[
            acquisition_available_df["recommended_use"] == "dev_pool_candidate"
        ]
        .sort_values("doc_id")
        .reset_index(drop=True)
        .copy()
    )

    if len(candidates) == 0:
        return None

    selected = candidates.sample(
        n=1,
        random_state=random_seed
    )

    return selected.iloc[0]["doc_id"]


In [ ]:
def select_purchase_by_bin(
    purchase_available_df,
    complexity_bin,
    n_docs,
    excluded_ids=None,
    random_seed=1
):
    """
    Select deterministic Purchase/Sale documents from a complexity bin.

    Parameters
    ----------
    purchase_available_df : pandas.DataFrame
        Candidate Purchase/Sale documents.
    complexity_bin : str
        Complexity group to sample from:
        low, medium, or high.
    n_docs : int
        Number of documents to sample.
    excluded_ids : set or None
        Documents excluded from selection.
    random_seed : int
        Random seed controlling deterministic sampling.

    Returns
    -------
    list
        Sorted list of selected document identifiers.
    """
    excluded_ids = set(excluded_ids or [])

    candidates = (
        purchase_available_df[
            (purchase_available_df["complexity_bin"] == complexity_bin)
            & (~purchase_available_df["doc_id"].isin(excluded_ids))
        ]
        .sort_values("doc_id")
        .reset_index(drop=True)
        .copy()
    )

    if len(candidates) == 0:
        return []

    selected = candidates.sample(
        n=min(n_docs, len(candidates)),
        random_state=random_seed
    )

    return sorted(selected["doc_id"].tolist())

# Development pool selection
Builds the dev_pool split using deterministic seed-based selection for Acquisition and Purchase documents.

In [ ]:
selected_acquisition_dev_pool = select_acquisition_for_dev_pool(
    acquisition_available_df,
    random_seed=13
)

print("\n=== SELECTED ACQUISITION FOR DEV_POOL ===")
print(selected_acquisition_dev_pool)

selected_low_purchase_dev_pool = select_purchase_by_bin(
    purchase_available_df,
    complexity_bin="low",
    n_docs=2,
    random_seed=117
)

print("\n=== SELECTED LOW PURCHASE DOCS FOR DEV_POOL ===")
print(selected_low_purchase_dev_pool)

selected_medium_purchase_dev_pool = select_purchase_by_bin(
    purchase_available_df,
    complexity_bin="medium",
    n_docs=3,
    random_seed=7007
)

print("\n=== SELECTED MEDIUM PURCHASE DOCS FOR DEV_POOL ===")
print(selected_medium_purchase_dev_pool)

selected_high_purchase_dev_pool = select_purchase_by_bin(
    purchase_available_df,
    complexity_bin="high",
    n_docs=3,
    random_seed=6340
)

print("\n=== SELECTED HIGH PURCHASE DOCS FOR DEV_POOL ===")
print(selected_high_purchase_dev_pool)

selected_purchase_dev_pool = (
    selected_low_purchase_dev_pool
    + selected_medium_purchase_dev_pool
    + selected_high_purchase_dev_pool
)

dev_pool_ids = set(
    [selected_acquisition_dev_pool]
    + selected_purchase_dev_pool
)

print("\n=== DEV_POOL IDS EXCLUDED FROM DEV_TUNE ===")
print(sorted(dev_pool_ids))

# Development tuning selection
Builds the dev_tune split while excluding documents already assigned to dev_pool.

In [ ]:
selected_low_purchase_dev_tune = select_purchase_by_bin(
    purchase_available_df,
    complexity_bin="low",
    n_docs=3,
    excluded_ids=dev_pool_ids,
    random_seed=9375
)

selected_medium_purchase_dev_tune = select_purchase_by_bin(
    purchase_available_df,
    complexity_bin="medium",
    n_docs=4,
    excluded_ids=dev_pool_ids,
    random_seed=197093
)

selected_high_purchase_dev_tune = select_purchase_by_bin(
    purchase_available_df,
    complexity_bin="high",
    n_docs=3,
    excluded_ids=dev_pool_ids,
    random_seed=4408
)

selected_purchase_dev_tune = (
    selected_low_purchase_dev_tune
    + selected_medium_purchase_dev_tune
    + selected_high_purchase_dev_tune
)

dev_tune_ids = set(selected_purchase_dev_tune)

print("\n=== SELECTED LOW PURCHASE DOCS FOR DEV_TUNE ===")
print(selected_low_purchase_dev_tune)

print("\n=== SELECTED MEDIUM PURCHASE DOCS FOR DEV_TUNE ===")
print(selected_medium_purchase_dev_tune)

print("\n=== SELECTED HIGH PURCHASE DOCS FOR DEV_TUNE ===")
print(selected_high_purchase_dev_tune)

print("\n=== SELECTED PURCHASE DOCS FOR DEV_TUNE ===")
print(selected_purchase_dev_tune)

# Final split generation and export
Constructs the final split configuration, validates split integrity, and exports all experimental artifacts.

## Final split construction

In [ ]:
final_dev_pool_ids = sorted(dev_pool_ids)
final_dev_tune_ids = sorted(dev_tune_ids)

assigned_to_dev = set(final_dev_pool_ids) | set(final_dev_tune_ids)

final_test_ids = sorted(
    set(docs_df["doc_id"]) - assigned_to_dev
)

split = {
    "test": final_test_ids,
    "dev_tune": final_dev_tune_ids,
    "dev_pool": final_dev_pool_ids,
}

## Split validation

In [ ]:
print("\n=== SPLIT SIZES ===")

for split_name, ids in split.items():
    print(split_name, len(ids))

print("\n=== OVERLAPS ===")

print(
    "dev_pool / dev_tune:",
    len(set(split["dev_pool"]) & set(split["dev_tune"]))
)

print(
    "dev_pool / test:",
    len(set(split["dev_pool"]) & set(split["test"]))
)

print(
    "dev_tune / test:",
    len(set(split["dev_tune"]) & set(split["test"]))
)

all_split_ids = (
    set(split["dev_pool"])
    | set(split["dev_tune"])
    | set(split["test"])
)

all_doc_ids = set(docs_df["doc_id"])

print("\n=== COVERAGE ===")

print("All docs covered:", all_split_ids == all_doc_ids)

missing_docs = sorted(all_doc_ids - all_split_ids)

print("Missing docs:", missing_docs)

## Export split definitions

In [ ]:
split_json = {
    split_name: sorted(ids)
    for split_name, ids in split.items()
}

with open(OUT_DIR / "split.json", "w", encoding="utf-8") as f:
    json.dump(
        split_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nSaved split.json")

for split_name, ids in split.items():

    output_path = OUT_DIR / f"{split_name}.txt"

    with open(output_path, "w", encoding="utf-8") as f:

        for doc_id in sorted(ids):
            f.write(doc_id + "\n")

    print("Saved:", output_path)

## Export diagnostic tables

In [ ]:
summary_rows = []

for split_name, ids in split.items():

    sub_df = docs_df[
        docs_df["doc_id"].isin(ids)
    ]

    summary_rows.append({
        "split": split_name,
        "n_docs": len(sub_df),
        "mean_people": round(sub_df["n_people"].mean(), 2),
        "mean_places": round(sub_df["n_places"].mean(), 2),
        "mean_events": round(sub_df["n_events"].mean(), 2),
        "mean_complexity": round(sub_df["complexity_score"].mean(), 2),
    })

split_summary_df = pd.DataFrame(summary_rows)

split_summary_df.to_csv(
    OUT_DIR / "split_summary.csv",
    index=False
)

print("Saved split_summary.csv")

docs_df.sort_values("doc_id").to_csv(
    OUT_DIR / "documents_with_event_types.csv",
    index=False
)

print("Saved documents_with_event_types.csv")

## Copy split JSON files

In [ ]:
import shutil

SPLIT_JSON_DIR = OUT_DIR / "split_jsons"
SPLIT_JSON_DIR.mkdir(exist_ok=True)

for split_name, ids in split.items():

    target_dir = SPLIT_JSON_DIR / split_name
    target_dir.mkdir(exist_ok=True)

    for doc_id in sorted(ids):

        src = files[doc_id]
        dst = target_dir / src.name

        shutil.copy2(src, dst)

    print("Copied JSONs for:", split_name)


print("\n=== DONE ===")
print("Outputs saved to:", OUT_DIR)